***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.2 第一代校准（1GC）](8_2_1gc.ipynb)
    * 下一节： [8.4 第三代校准（3GC）](8_4_3gc.ipynb)

***

导入标准模块:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

导入本节所需的专用模块:

In [ ]:
pass

In [ ]:
HTML('../style/code_toggle.html')

## 8.3 第二代校准（2GC）：方向无关自校准

完成 1GC 之后，也就是把由校准源求得的天线增益应用到目标场之后，我们通常已经可以得到一幅还算不错的目标场图像。但如果进一步采用*自校准*（self-calibration）框架，这幅图像的动态范围往往还能显著提升。

下面我们分别在[$\S$ 8.3.1 &#10549;](#cal:sec:selfcal)中介绍自校准本身，并在[$\S$ 8.3.2 &#10549;](#cal:sec:hybrid_mapping)中介绍混合成图（hybrid-mapping），后者可以看作是自校准的前身。

### 8.3.1 自校准 <a id='cal:sec:selfcal'></a> <!--\label{cal:sec:selfcal}-->

自校准的核心思想，是直接利用目标场本身的观测数据来校准可见度。图 [8.3.1 &#10549;](#cal:fig:self_cal) 给出了一个自校准框架的示意图。整个过程在两个域之间反复切换：图像域和可见度域。图像域里主要进行去卷积和源提取；可见度域里则执行校准。关于去卷积和源探测，可分别参考[$\S$ 6.2 &#10142;](../6_Deconvolution/6_2_clean.ipynb)和[$\S$ 6.5 &#10142;](../6_Deconvolution/6_5_source_finding.ipynb)。

需要注意的是，图 [8.3.1 &#10549;](#cal:fig:self_cal) 只是对自校准框架的一个非常简化的表示。一个典型的自校准流程大致如下：

<p class=conclusion>
  <font size=4> <b>自校准算法</b></font>
  <br>
  <br>
1. 先根据 1GC 之后的图像，为目标场建立一个不完整的初始天空模型。<br>
2. 利用当前的初始或更新后的天空模型去校准观测到的可见度，并对校准后的数据重新成像。<br>
3. 对新得到的图像执行去卷积。<br>
4. 在去卷积后的图像上运行源提取器，构造一个更精确的天空模型。<br>
5. 返回第 2 步继续迭代；若已经达到目标动态范围，或进一步改进已不再明显，则终止算法。<br>
</p>

<img src='figures/Selfcal2.png' width=80%>

**图 8.3.1**：自校准框架。<a id='cal:fig:self_cal'></a> <!--\label{cal:fig:self_cal}-->

下面给出几幅射电图像，用来展示自校准的实际效果。图中的图像都对应 LOFAR 对北天极（NCP）的观测。图 [8.3.2 &#10549;](#cal:fig:1GC_lofar) 是在仅应用校准源解之后得到的结果；图 [8.3.3 &#10549;](#cal:fig:2GC_lofar) 则是对目标场进行自校准（并追加了一些数据标记）之后得到的图像；图 [8.3.4 &#10549;](#cal:fig:2GC_lofar_clip) 则是在图 [8.3.3 &#10549;](#cal:fig:2GC_lofar) 的基础上进一步压缩颜色条范围，以便显示更弱源。

比较这三幅图，可以观察到：

* 射电星系 3C 220.3 只有在执行自校准之后才变得清晰可见；
* 射电星系 3C 61.1 在自校准之后明显变亮，也更接近其真实的视流量；
* 一些更弱的源在自校准之后才被显现出来，尤其在图 [8.3.4 &#10549;](#cal:fig:2GC_lofar_clip) 中更明显。

简而言之，自校准能够显著提升射电图像的质量。

<img src='figures/1GC_only.png' width=80%>

**图 8.3.2**：执行自校准之前的 NCP 图像，此时只进行了 1GC 校准。该观测来自 LOFAR。<a id='cal:fig:1GC_lofar'></a> <!--\label{cal:fig:1GC_lofar}-->

<img src='figures/2GC.png' width=80%>

**图 8.3.3**：执行自校准之后的 NCP 图像。<a id='cal:fig:2GC_lofar'></a> <!--\label{cal:fig:2GC_lofar}-->

<img src='figures/2GC_clip.png' width=80%>

**图 8.3.4**：执行自校准之后的 NCP 图像。这里对颜色条范围做了裁剪，以便更清楚地显示弱源。<a id='cal:fig:2GC_lofar_clip'></a> <!--\label{cal:fig:2GC_lofar_clip}-->

### 8.3.2 混合成图（Hybrid-mapping） <a id='cal:sec:hybrid_mapping'></a> <!--\label{cal:sec:hybrid_mapping}-->

到目前为止，我们还没有专门讨论在自校准框架内部到底该使用哪一种校准算法。这是因为，从“框架”角度看，具体算法并不是最核心的部分。不过值得指出的是，如今最常用的还是[$\S$ 8.1 &#10142;](../8_Calibration/8_1_calibration_least_squares_problem.ipynb#cal:sec:cal_ls)中介绍的最小二乘方法。但历史上并非一直如此。

在更早期的自校准工作中，人们实际上更多依赖闭合量（closure quantities）（参见 [<cite data-cite='Jennison1958'>A phase sensitive interferometer technique for the measurement of the Fourier transforms of spatial brightness distributions of small angular extent</cite> &#10548;](http://mnras.oxfordjournals.org/content/118/3/276.short) 和 [<cite data-cite='Smith1952'>The measurement of the angular diameter of radio stars</cite> &#10548;](http://iopscience.iop.org/article/10.1088/0370-1301/65/12/309/meta;jsessionid=9DF36DAB88B75643A607FA921E11CC4A.c2.iopscience.cld.iop.org)）。当时，如果图 [8.3.1 &#10549;](#cal:fig:self_cal) 中校准子模块采用的是闭合量方法，那么这个过程通常不叫“self-calibration”，而叫作*hybrid-mapping*。

<div class=warn>
<b>注意：</b> 一般来说，如果图 [8.3.1 &#10549;](#cal:fig:self_cal) 中的校准求解器使用的是最小二乘法，我们通常称之为“自校准”；若采用的是闭合量方法，则更常称之为“混合成图”。
</div>

我们已在[$\S$ 8.2 &#10142;](../8_Calibration/8_2_1gc.ipynb#cal:sec:closure)中介绍过闭合量。最著名的 hybrid-mapping 方法之一，由 Readhead 和 Wilkinson 提出（[<cite data-cite='Readhead1978'>The mapping of compact radio sources from VLBI data</cite> &#10548;](http://adsabs.harvard.edu/full/1978ApJ...223...25R)）。其基本步骤为：

<p class=conclusion>
  <font size=4> <b>Hybrid-mapping 算法</b></font>
  <br>
  <br>
  1. 对于一个由 $N$ 个天线组成的阵列，先从当前初始或更新后的模型可见度中得到 $N-1$ 个基线相位。<br>
  2. 调整这些基线相位，使闭合相位尽可能最小。<br>
  3. 用校正后的可见度成像，并执行去卷积。<br>
  4. 根据去卷积后的图像更新天空模型。<br>
  5. 返回第 1 步继续迭代，或者在达到收敛时终止。<br>
</p>

<br>
<div class=advice>
<b>说明：</b> *冗余校准*（redundant calibration，见 [<cite data-cite='Noordam1982'>High dynamic range mapping of strong radio sources, with application to 3C84</cite> &#10548;](http://adsabs.harvard.edu/abs/1982Natur.299..597N)）同样依赖闭合量，但它并不需要天空模型，而是利用阵列中重复测量同一 $uv$ 点的冗余性，只需让这些测量彼此一致即可。
</div>

最小二乘校准最早由 [<cite data-cite='Schwab1980'>Adaptive calibration of radio interferometer data</cite> &#10548;](http://proceedings.spiedigitallibrary.org/proceeding.aspx?articleid=1229965) 和 [<cite data-cite='Cornwill1981'>A new method for making maps with unstable radio interferometers</cite> &#10548;](http://mnras.oxfordjournals.org/content/196/4/1067.short) 提出。最小二乘方法的一个重要优点，是它允许我们直接求解每面天线的增益，而不是基于基线求解增益。关于自校准历史与方法的综述，可参考 [<cite data-cite='Ekers1984'>The almost serendipitous discovery of self-calibration</cite> &#10548;](http://adsabs.harvard.edu/abs/1984sdra.conf..154E) 和 [<cite data-cite='Pearson1984'>Image formation by self-calibration in radio astronomy</cite> &#10548;](http://adsabs.harvard.edu/full/1984ARA%26A..22...97P)。

<br>
<div class=advice>
<b>说明：</b> 从更广义的角度看，自校准也可以被视为 *Gerchberg-Saxton* 算法的一种变体（见 [<cite data-cite='Gerchberg1972'>A practical algorithm for the determination of phase from image and diffraction plane pictures</cite> &#10548;](http://www.u.arizona.edu/~ppoon/GerchbergandSaxton1972.pdf)）。
</div>

***

下一节： [8.4 第三代校准（3GC）](8_4_3gc.ipynb)